# DataMate — So sánh Qwen2.5-14B-Instruct-1M Base vs LoRA trên Colab (Pipeline P1/P2/P3)

Notebook chấm cùng **30 câu Unseen Dev** theo thứ tự **Base (không LoRA)** -> **Base + LoRA**, giữ nguyên toàn văn `rulebook.md` trong `system_prompt`, `user_prompt` sạch chữ `rulebook`, đồng thời bật đầy đủ:
- **P1 Schema**: Hiển thị `[TOÀN BỘ NHÃN / ALL LABELS]` (tối đa 25 nhãn) cho cột ít giá trị, tách riêng `[MẪU THAM KHẢO / SAMPLE VALUES]` cho cột nhiều giá trị.
- **P2 Guardrails + Abstain Gate**: Chặn kết quả `0` thiếu bằng chứng, `[0, 0]`, bảng nợ `<= 0`, top-k trùng lặp và chuyển `unanswerable` rủi ro sang `abstain`.
- **P3 Sandbox & Temporal fixes**: Đầy đủ helper `unmerge_and_ffill`, `load_excel_unmerge` và mốc thời gian tương đối.

### Cách chạy nhanh trên Colab
1. Chọn **Runtime -> Change runtime type -> G4 GPU** (hoặc **A100 / L4 / T4**).
2. Chọn **Run All** — notebook tự tải `r1_reasoning_kaggle_bundle.zip` bản mới nhất từ `Crinkk/Zalo-challenge` (đã gồm code P1/P2/P3, `data/train/` và 30 câu Unseen Dev).
3. Cell cuối in bảng đối chứng PA, số lượt `observation`, số lần kích hoạt `guardrails` và `ΔPA = PA adapter − base`.


In [ ]:
from datetime import datetime, timezone
import os
from pathlib import Path
import subprocess

# Token dùng cho bản handoff riêng. Có thể xóa giá trị để dùng Colab Secret HF_TOKEN.
HF_TOKEN_MANUAL = 'REDACTED_HF_TOKEN'


def resolve_hf_token() -> str:
    if HF_TOKEN_MANUAL.strip():
        return HF_TOKEN_MANUAL.strip()
    env_tok = os.environ.get('HF_TOKEN', '').strip()
    if env_tok:
        return env_tok
    try:
        from google.colab import userdata
        sec = (userdata.get('HF_TOKEN') or '').strip()
        if sec:
            return sec
    except Exception:
        pass
    try:
        from huggingface_hub import get_token
        cli_tok = (get_token() or '').strip()
        if cli_tok:
            return cli_tok
    except Exception:
        pass
    return ''


HF_TOKEN = resolve_hf_token()
if HF_TOKEN:
    print('🔑 Đã nhận diện HF_TOKEN.')
else:
    print('⚠️ Chưa thấy HF_TOKEN. Cần token nếu bundle hoặc adapter là repo riêng tư.')

# Nguồn dữ liệu & model
BUNDLE_REPO = 'Crinkk/Zalo-challenge'
BUNDLE_REPO_TYPE = 'model'
BUNDLE_FILENAME = 'r1_reasoning_kaggle_bundle.zip'
CODE_OVERLAY_ZIP = ''  # Tùy chọn: trỏ tới file zip local nếu muốn đè code mới mà không qua HF

BASE_MODEL_ID = 'Qwen/Qwen2.5-14B-Instruct-1M'
HF_REPO = 'Crinkk/datamate-qwen25-14b-1m-reasoning-adapters'
# Có thể điền subpath trên HF_REPO hoặc đường dẫn thư mục local trên Colab (ví dụ /content/runs/.../checkpoint-xxx)
ADAPTER_SUBPATH = 'qwen25_14b_1m_code_only_rulebook_20261111_155610/final_adapter'

# Cấu hình lượt chạy đối chứng
RUN_BASE = True
RUN_ADAPTER = True
ABSTAIN_GATE = True            # True: đổi unanswerable rủi ro -> abstain (0 điểm thay vì -1 điểm)
REQUIRE_NEW_PIPELINE = True    # True: dừng ngay nếu bundle thiếu code P1/P2/P3 mới
UPLOAD_RESULTS = True
AUTO_DISCONNECT = False        # Đặt True nếu muốn tự ngắt runtime Colab sau khi upload xong HF.

# Thư mục làm việc trên SSD cục bộ của Colab (/content)
WORK_DIR = Path('/content')
MODELS_DIR = WORK_DIR / 'models'
LOCAL_MODEL_DIR = MODELS_DIR / 'Qwen2.5-14B-Instruct-1M'
ADAPTERS_DIR = WORK_DIR / 'adapters'


def detect_gpus() -> list[dict]:
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader,nounits'],
        text=True,
    )
    gpus = []
    for line in out.strip().splitlines():
        parts = [p.strip() for p in line.split(',')]
        if len(parts) < 3:
            continue
        gpus.append({
            'name': parts[0],
            'vram_gb': float(parts[1]) / 1024.0,
            'compute_cap': parts[2],
        })
    return gpus


def select_eval_profile(gpus: list[dict]) -> dict:
    assert gpus, '❌ Không tìm thấy GPU NVIDIA! Hãy chọn Runtime type -> G4 GPU (hoặc A100/L4) rồi chạy lại.'
    gpu = gpus[0]
    name_upper = gpu['name'].upper()
    vram_gb = gpu['vram_gb']
    if vram_gb >= 80.0:
        tier = 'g4_96gb' if ('PRO 6000' in name_upper or vram_gb >= 90.0) else 'a100_80gb'
        return {
            'gpu_tier': tier,
            'backend': 'vllm',
            'dtype': 'bfloat16',
            'load_in_4bit': False,
            'max_model_len': 16384,
            'max_tokens': 4096,
            'gpu_mem_util': 0.90,
            'max_num_seqs': 16,
            'enforce_eager': False,
            'enable_prefix_caching': True,
            'tp_size': 1,
        }
    if vram_gb >= 38.0:
        return {
            'gpu_tier': 'a100_40gb',
            'backend': 'vllm',
            'dtype': 'bfloat16',
            'load_in_4bit': False,
            'max_model_len': 16384,
            'max_tokens': 4096,
            'gpu_mem_util': 0.90,
            'max_num_seqs': 2,
            'enforce_eager': False,
            'enable_prefix_caching': True,
            'tp_size': 1,
        }
    if vram_gb >= 20.0:
        return {
            'gpu_tier': 'l4_22gb',
            'backend': 'vllm',
            'dtype': 'float16',
            'load_in_4bit': True,
            'max_model_len': 12288,
            'max_tokens': 3072,
            'gpu_mem_util': 0.90,
            'max_num_seqs': 2,
            'enforce_eager': False,
            'enable_prefix_caching': True,
            'tp_size': 1,
        }
    return {
        'gpu_tier': 't4_15gb',
        'backend': 'vllm',
        'dtype': 'float16',
        'load_in_4bit': True,
        'max_model_len': 12288,
        'max_tokens': 3072,
        'gpu_mem_util': 0.90,
        'max_num_seqs': 1,
        'enforce_eager': True,
        'enable_prefix_caching': False,
        'tp_size': 1,
    }


GPUS = detect_gpus()
PROFILE = select_eval_profile(GPUS)
GPU_NAME = GPUS[0]['name']
VRAM_GB = GPUS[0]['vram_gb']
COMPUTE_CAP = GPUS[0]['compute_cap']

GPU_TIER = PROFILE['gpu_tier']
BACKEND = PROFILE['backend']
DTYPE = PROFILE['dtype']
LOAD_IN_4BIT = PROFILE['load_in_4bit']
MAX_MODEL_LEN = PROFILE['max_model_len']
MAX_TOKENS = PROFILE['max_tokens']
GPU_MEM_UTIL = PROFILE['gpu_mem_util']
MAX_NUM_SEQS = PROFILE['max_num_seqs']
ENFORCE_EAGER = PROFILE['enforce_eager']
ENABLE_PREFIX_CACHING = PROFILE['enable_prefix_caching']
TP_SIZE = PROFILE['tp_size']

print('=' * 68)
print(f'🖥️  GPU phát hiện   : {len(GPUS)}x {GPU_NAME} ({VRAM_GB:.1f} GB VRAM | sm_{COMPUTE_CAP.replace(".", "")})')
print(f'⚡ Profile chọn    : {GPU_TIER} | dtype={DTYPE} | 4-bit={LOAD_IN_4BIT}')
print(f'📦 Batch & Context : max_num_seqs={MAX_NUM_SEQS} | max_model_len={MAX_MODEL_LEN} | max_tokens={MAX_TOKENS}')
print(f'🧠 vLLM flags      : TP={TP_SIZE} | prefix_caching={ENABLE_PREFIX_CACHING} | cuda_graphs={not ENFORCE_EAGER}')
print(f'🎯 Runs & Gates    : RUN_BASE={RUN_BASE} | RUN_ADAPTER={RUN_ADAPTER} | ABSTAIN_GATE={ABSTAIN_GATE}')
print('=' * 68)


In [ ]:
import subprocess
import sys

# Lưu ý: Không import torch/vllm trực tiếp vào kernel Colab ở đây để tránh xung đột thư viện C++ khi nâng cấp pip;
# mọi bước kiểm tra kernel CUDA sm_120 đều chạy trong subprocess sạch.
print(f'🐍 Python runtime: {sys.version.split()[0]}')

VERIFY_CODE = f'''
import torch
import vllm
import transformers
import peft
import duckdb
import openpyxl
import python_calamine
import xlrd
import yaml
import pandas
import numpy
import pydantic
import huggingface_hub
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

assert torch.cuda.is_available(), "CUDA unavailable"
dt = torch.bfloat16 if {DTYPE!r} == "bfloat16" else torch.float16
a = torch.randn(64, 64, device="cuda", dtype=dt)
c = (a @ a.T).sum().item()
torch.cuda.synchronize()
cap = torch.cuda.get_device_capability(0)
print(
    f"OK torch={{torch.__version__}} cuda={{torch.version.cuda}} "
    f"vllm={{vllm.__version__}} transformers={{transformers.__version__}} "
    f"cap={{cap}} matmul={{c:.2f}}"
)
'''


def verify_runtime() -> tuple[int, str]:
    res = subprocess.run(
        [sys.executable, '-c', VERIFY_CODE],
        text=True,
        capture_output=True,
    )
    out = ((res.stdout or '') + '\n' + (res.stderr or '')).strip()
    return res.returncode, out


def run_pip(args: list[str]) -> None:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *args]
    print('📦', ' '.join(cmd[:8]), f'... (+{max(0, len(cmd) - 8)} args)')
    subprocess.run(cmd, check=True)


rc, out = verify_runtime()
if rc == 0:
    print(f'✅ Môi trường đã sẵn sàng: {out.splitlines()[-1]}')
else:
    print('🔧 Đang cài đặt vLLM (hỗ trợ Blackwell sm_120) và thư viện xử lý bảng...')
    run_pip(['-U', 'vllm'])
    pkgs = [
        'pandas>=2.2.0',
        'numpy>=1.26.0',
        'openpyxl>=3.1.2',
        'python-calamine>=0.2.0',
        'xlrd>=2.0.1',
        'duckdb>=1.1.0',
        'pyyaml>=6.0.1',
        'tqdm>=4.66.0',
        'pydantic>=2.7.0',
        'huggingface-hub>=0.26.0',
        'hf_transfer>=0.1.8',
        'safetensors>=0.4.5',
        'peft>=0.13.0',
        'accelerate>=1.0.0',
        'sentencepiece',
    ]
    if LOAD_IN_4BIT:
        pkgs.append('bitsandbytes>=0.45.0')
    run_pip(pkgs)

    rc, out = verify_runtime()
    if rc != 0:
        print('⚠️ Lần kiểm tra đầu gặp lỗi tương thích CUDA/torch, thử đồng bộ lại với cu128 wheels...')
        print(out[-1200:])
        run_pip([
            '-U',
            '--upgrade-strategy', 'eager',
            'vllm',
            'torch',
            'torchvision',
            'torchaudio',
            '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
        ])
        rc, out = verify_runtime()

    if rc != 0:
        raise RuntimeError(f'❌ Kiểm tra vLLM / CUDA kernel thất bại:\n{out[-2500:]}')
    print(f'✅ Cài đặt & kiểm tra CUDA kernel thành công: {out.splitlines()[-1]}')


In [ ]:
import importlib.util
import json
import os
import tempfile
import zipfile
from huggingface_hub import hf_hub_download, snapshot_download

HAS_HF_TRANSFER = importlib.util.find_spec('hf_transfer') is not None
if HAS_HF_TRANSFER:
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'


def safe_hf_call(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as exc:
        if os.environ.get('HF_HUB_ENABLE_HF_TRANSFER') == '1':
            print(f'⚠️ hf_transfer gặp lỗi ({type(exc).__name__}), thử tải lại bằng HTTP tiêu chuẩn...')
            os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
            return fn(*args, **kwargs)
        raise


def safe_extract_zip(zip_path: Path, dest_dir: Path) -> int:
    dest_root = dest_dir.resolve()
    count = 0
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for member in zf.infolist():
            if member.is_dir():
                continue
            target = (dest_root / member.filename).resolve()
            if not target.is_relative_to(dest_root):
                raise ValueError(f'Unsafe zip path: {member.filename}')
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member, 'r') as src_f, target.open('wb') as dst_f:
                dst_f.write(src_f.read())
            count += 1
    return count


DL_TOKEN = HF_TOKEN or None

# 1. Tải bundle dữ liệu + 30 câu Unseen Dev + code nền từ Hugging Face
bundle_zip = safe_hf_call(
    hf_hub_download,
    repo_id=BUNDLE_REPO,
    filename=BUNDLE_FILENAME,
    repo_type=BUNDLE_REPO_TYPE,
    token=DL_TOKEN,
)
REPO_DIR = Path(tempfile.mkdtemp(prefix='datamate_repo_', dir=WORK_DIR))
base_files = safe_extract_zip(Path(bundle_zip), REPO_DIR)
print(f'📦 Đã giải nén {base_files} file từ {BUNDLE_REPO}/{BUNDLE_FILENAME} vào {REPO_DIR}.')

# 1b. Ghi đè code mới (src/, scripts/, configs/) từ file overlay nếu có
overlay_path = Path(CODE_OVERLAY_ZIP) if CODE_OVERLAY_ZIP else None
if overlay_path and overlay_path.is_file():
    overlaid = safe_extract_zip(overlay_path, REPO_DIR)
    print(f'🔄 Đã nạp {overlaid} file code mới từ {overlay_path} (giữ nguyên data/ và dev/ của bundle).')
else:
    print(f'ℹ️ Không thấy {CODE_OVERLAY_ZIP}; dùng trực tiếp code bên trong bundle {BUNDLE_FILENAME}.')

QUESTIONS_FILE = REPO_DIR / 'dev' / 'questions.jsonl'
GOLD_FILE = REPO_DIR / 'dev' / 'gold.jsonl'
DATA_DIR = REPO_DIR / 'data' / 'train'
assert QUESTIONS_FILE.exists(), f'❌ Thiếu {QUESTIONS_FILE}'
assert GOLD_FILE.exists(), f'❌ Thiếu {GOLD_FILE}'
assert DATA_DIR.exists(), f'❌ Thiếu {DATA_DIR}'
q_count = sum(1 for ln in QUESTIONS_FILE.read_text(encoding='utf-8').splitlines() if ln.strip())
assert q_count == 30, f'Bundle phải chứa 30 câu Unseen Dev, nhận được {q_count}.'
print(f'✅ Dữ liệu eval sẵn sàng ({q_count} câu Unseen Dev).')

# 1c. Kiểm tra chắc chắn repo đang chứa pipeline P1/P2/P3 mới (tránh chạy nhầm code cũ trên HF)
eval_script_txt = (REPO_DIR / 'scripts' / 'evaluate_pilot_30.py').read_text(encoding='utf-8')
solver_txt = (REPO_DIR / 'src' / 'agent' / 'solver.py').read_text(encoding='utf-8')
loader_txt = (REPO_DIR / 'src' / 'profiler' / 'business_loader.py').read_text(encoding='utf-8')
prompt_path = REPO_DIR / 'src' / 'agent' / 'prompts.py'
prompt_source = prompt_path.read_text(encoding='utf-8')

missing_features = []
for flag in ('rulebook_in_system', 'use_eval_schema', 'enable_guardrails'):
    if flag not in eval_script_txt or flag not in solver_txt:
        missing_features.append(flag)
if 'TOÀN BỘ NHÃN / ALL LABELS' not in loader_txt:
    missing_features.append('P1_exhaustive_labels')
if 'rulebook_in_system' not in prompt_source:
    missing_features.append('prompts_rulebook_in_system')

if missing_features:
    msg = (
        f'❌ Code trong {REPO_DIR} là bản CŨ, thiếu: {missing_features}. '
        f'Hãy upload file result/datamate_eval_code_overlay.zip lên {CODE_OVERLAY_ZIP} '
        'hoặc chạy scripts/package_and_upload_bundle.py để cập nhật bundle trên Hugging Face trước khi chạy eval!'
    )
    if REQUIRE_NEW_PIPELINE:
        raise RuntimeError(msg)
    print('⚠️', msg)
else:
    print('✅ Đã xác minh pipeline P1/P2/P3 + rulebook_in_system bản mới nhất.')

rulebook_text = (DATA_DIR / 'rulebook.md').read_text(encoding='utf-8').strip()
assert rulebook_text, 'Bundle thiếu nội dung rulebook.'

# Tương thích ngược nếu chạy fallback bản prompts cũ không có rulebook_in_system
if 'rulebook_in_system' not in prompt_source:
    old_dates = (
        '        if matches:\n'
        '            dates.append(f"- {label}: {value}")'
    )
    new_dates = (
        '        if matches and not label.startswith("quy ước ca làm việc"):\n'
        '            display_label = label.split(" (", 1)[0] if "rulebook" in label.casefold() else label\n'
        '            dates.append(f"- {display_label}: {value}")'
    )
    assert prompt_source.count(old_dates) == 1, 'Bundle đã đổi cách tạo user prompt. Kiểm tra prompts.py.'
    prompt_source = prompt_source.replace(old_dates, new_dates).replace(
        'Mốc thời gian (hai đầu đều tính):', 'Mốc thời gian:'
    )
    compile(prompt_source, str(prompt_path), 'exec')
    prompt_path.write_text(prompt_source, encoding='utf-8')

# 2. Tải Base Model Qwen2.5-14B-Instruct-1M về SSD Colab (/content)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'⬇️ Đang tải Base Model {BASE_MODEL_ID} về {LOCAL_MODEL_DIR}...')
safe_hf_call(
    snapshot_download,
    repo_id=BASE_MODEL_ID,
    local_dir=str(LOCAL_MODEL_DIR),
    token=DL_TOKEN,
)

# Gỡ cấu hình Dual Chunk / Sparse Attention để vLLM chạy Full Attention + LoRA ổn định ở ctx 16k
cfg_path = LOCAL_MODEL_DIR / 'config.json'
cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
if cfg.pop('dual_chunk_attention_config', None) is not None:
    cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding='utf-8')
    print('🔧 Đã gỡ dual_chunk_attention_config trong bản local config.json.')
sparse_cfg = LOCAL_MODEL_DIR / 'sparse_attention_config.json'
if sparse_cfg.exists():
    sparse_cfg.unlink()
    print('🔧 Đã gỡ sparse_attention_config.json trong bản local.')
print('✅ Base Model sẵn sàng.')


In [ ]:
from collections import deque
import sys


def summarize_logs(logs: list[Path]) -> dict:
    obs_turns = 0
    obs_questions = 0
    guard_retries = 0
    p1_all_labels = 0
    p1_samples = 0
    for log_path in logs:
        trace = json.loads(log_path.read_text(encoding='utf-8'))
        attempts = trace.get('attempts') or []
        if attempts and attempts[0].get('prompt_messages'):
            u0 = attempts[0]['prompt_messages'][1]['content']
            if '[TOÀN BỘ NHÃN / ALL LABELS' in u0:
                p1_all_labels += 1
            if '[MẪU THAM KHẢO / SAMPLE VALUES' in u0:
                p1_samples += 1
        q_obs = sum(1 for a in attempts if a.get('kind') == 'observation')
        obs_turns += q_obs
        if q_obs > 0:
            obs_questions += 1
        guard_retries += sum(1 for a in attempts if a.get('guard_error'))
    return {
        'obs_turns': obs_turns,
        'obs_questions': obs_questions,
        'guard_retries': guard_retries,
        'p1_all_labels_questions': p1_all_labels,
        'p1_sample_questions': p1_samples,
    }


def run_eval(output_dir: Path, adapter_dir: Path | None = None) -> dict:
    output_dir.mkdir(parents=True, exist_ok=True)
    label = 'Base' if adapter_dir is None else 'Base + LoRA'
    print(f'🚀 {label} | {q_count} câu | {GPU_NAME} | {BACKEND} {DTYPE} | context={MAX_MODEL_LEN} | batch={MAX_NUM_SEQS}')

    cmd = [
        sys.executable, 'scripts/evaluate_pilot_30.py',
        '--base-model', str(LOCAL_MODEL_DIR),
        '--questions-file', str(QUESTIONS_FILE),
        '--gold-file', str(GOLD_FILE),
        '--data-dir', str(DATA_DIR),
        '--output-dir', str(output_dir),
        '--backend', BACKEND,
        '--dtype', DTYPE,
        '--tensor-parallel-size', str(TP_SIZE),
        '--gpu-memory-utilization', str(GPU_MEM_UTIL),
        '--max-num-seqs', str(MAX_NUM_SEQS),
        '--max-model-len', str(MAX_MODEL_LEN),
        '--max-tokens', str(MAX_TOKENS),
        '--prompt-format', 'system-user',
        '--prompt-variant', 'baseline',
    ]
    if not ABSTAIN_GATE:
        cmd.append('--no-abstain-gate')
    if adapter_dir is not None:
        cmd.extend(['--adapter-path', str(adapter_dir)])
    if LOAD_IN_4BIT:
        cmd.append('--load-in-4bit')
    if ENFORCE_EAGER:
        cmd.append('--enforce-eager')
    if not ENABLE_PREFIX_CACHING:
        cmd.append('--no-prefix-caching')

    env = os.environ.copy()
    env['PYTHONPATH'] = str(REPO_DIR)
    env['PYTHONUNBUFFERED'] = '1'
    env['VLLM_USE_V1'] = '1'
    env['VLLM_NO_USAGE_STATS'] = '1'
    env['TOKENIZERS_PARALLELISM'] = 'false'
    env.pop('VLLM_ATTENTION_BACKEND', None)
    env.pop('HF_TOKEN', None)

    with subprocess.Popen(
        cmd, cwd=str(REPO_DIR), env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    ) as proc:
        log_tail = deque(maxlen=60)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='', flush=True)
            log_tail.append(line)
        rc = proc.wait()

    assert rc == 0, f'❌ {label} thất bại (rc={rc})\n--- LAST 60 LINES ---\n' + ''.join(log_tail)
    report_path = output_dir / 'evaluation_report.json'
    submission_path = output_dir / 'submission.jsonl'
    assert report_path.is_file(), f'❌ Thiếu {report_path}'
    assert submission_path.is_file(), f'❌ Thiếu {submission_path}'
    report = json.loads(report_path.read_text(encoding='utf-8'))
    assert report['total_questions'] == q_count, f'❌ {label} chấm sai số câu: {report["total_questions"]}'
    actual_adapter = report.get('adapter_path')
    if adapter_dir is None:
        assert actual_adapter in (None, 'None', ''), f'❌ Base đã nạp adapter: {actual_adapter}'
    else:
        assert actual_adapter == str(adapter_dir.resolve()), f'❌ Đã nạp adapter khác: {actual_adapter}'

    logs = sorted((output_dir / 'logs').glob('*.json'))
    assert len(logs) == q_count, f'❌ {label} thiếu log để kiểm tra prompt.'
    for log_path in logs:
        trace = json.loads(log_path.read_text(encoding='utf-8'))
        system_msg, user_msg = trace['attempts'][0]['prompt_messages'][:2]
        assert system_msg['role'] == 'system' and rulebook_text in system_msg['content'], (
            f'❌ System prompt thiếu toàn văn rulebook trong {log_path.name}'
        )
        assert user_msg['role'] == 'user' and 'rulebook' not in user_msg['content'].casefold(), (
            f'❌ User prompt còn nhắc đến rulebook trong {log_path.name}'
        )

    diag = summarize_logs(logs)
    report['diagnostics'] = diag
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
    print(
        f"✅ {label}: PA = {report['penalized_accuracy']:+.4f} "
        f"({report['correct']} đúng / {report['wrong']} sai / {report['abstain']} abstain) | "
        f"obs={diag['obs_questions']} câu ({diag['obs_turns']} lượt) | "
        f"guard_retries={diag['guard_retries']} | {output_dir}"
    )
    return report


EVAL_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
EVAL_OUT_DIR = WORK_DIR / f'eval_unseen_30_colab_{GPU_TIER}_ctx{MAX_MODEL_LEN}_{EVAL_TIMESTAMP}'
BASE_REPORT = ADAPTER_REPORT = None
if RUN_BASE:
    BASE_REPORT = run_eval(EVAL_OUT_DIR / 'base')
else:
    print('⏭️ Bỏ qua lượt đánh giá Base (RUN_BASE=False).')


In [ ]:
ADAPTER_REPORT = None
if not RUN_ADAPTER:
    print('⏭️ Bỏ qua lượt đánh giá Adapter (RUN_ADAPTER=False).')
else:
    local_override = Path(ADAPTER_SUBPATH)
    if local_override.is_dir():
        EVAL_ADAPTER_DIR = local_override
        print(f'📂 Dùng adapter local tại {EVAL_ADAPTER_DIR}...')
    else:
        print(f'⬇️ Đang tải {HF_REPO}/{ADAPTER_SUBPATH}...')
        safe_hf_call(
            snapshot_download,
            repo_id=HF_REPO,
            allow_patterns=[
                f'{ADAPTER_SUBPATH}/adapter_config.json',
                f'{ADAPTER_SUBPATH}/adapter_model.safetensors',
                f'{ADAPTER_SUBPATH}/adapter_model.bin',
            ],
            local_dir=str(ADAPTERS_DIR),
            token=DL_TOKEN,
        )
        EVAL_ADAPTER_DIR = ADAPTERS_DIR / ADAPTER_SUBPATH

    assert (EVAL_ADAPTER_DIR / 'adapter_config.json').is_file(), f'❌ Thiếu adapter_config.json tại {EVAL_ADAPTER_DIR}'
    assert any((EVAL_ADAPTER_DIR / name).is_file() for name in ('adapter_model.safetensors', 'adapter_model.bin')), (
        f'❌ Thiếu trọng số LoRA tại {EVAL_ADAPTER_DIR}'
    )
    ADAPTER_REPORT = run_eval(EVAL_OUT_DIR / 'adapter', EVAL_ADAPTER_DIR)


In [ ]:
from huggingface_hub import HfApi

runs = [(name, rep) for name, rep in [('Base', BASE_REPORT), ('Base + LoRA', ADAPTER_REPORT)] if rep is not None]
assert runs, 'Chưa có lượt đánh giá nào hoàn tất (RUN_BASE và RUN_ADAPTER đều không chạy).'

print('=' * 76)
for name, report in runs:
    diag = report.get('diagnostics', {})
    unans = report.get('unanswerable', {})
    print(
        f"{name:12s} | PA {report['penalized_accuracy']:+.4f} | "
        f"{report['correct']} đúng / {report['wrong']} sai / {report['abstain']} abstain | "
        f"turns={report.get('model_turns')} | obs={diag.get('obs_questions', 0)} câu ({diag.get('obs_turns', 0)} lượt) | "
        f"guard_retries={diag.get('guard_retries', 0)} | unans={unans}"
    )

delta_pa = None
if BASE_REPORT is not None and ADAPTER_REPORT is not None:
    delta_pa = ADAPTER_REPORT['penalized_accuracy'] - BASE_REPORT['penalized_accuracy']
    print(f'ΔPA (adapter − base): {delta_pa:+.4f}')
print('=' * 76)

comparison = {
    'base_model': BASE_MODEL_ID,
    'adapter_repo': HF_REPO,
    'adapter_subpath': ADAPTER_SUBPATH,
    'pipeline': {
        'rulebook_in_system': True,
        'use_eval_schema': True,
        'enable_guardrails': True,
        'abstain_gate': ABSTAIN_GATE,
    },
    'base': BASE_REPORT,
    'adapter': ADAPTER_REPORT,
    'delta_pa': delta_pa,
}
comparison_path = EVAL_OUT_DIR / 'comparison.json'
comparison_path.parent.mkdir(parents=True, exist_ok=True)
comparison_path.write_text(json.dumps(comparison, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'📁 Report, submission, logs và comparison.json: {EVAL_OUT_DIR}')

if UPLOAD_RESULTS:
    upload_token = HF_TOKEN or resolve_hf_token()
    assert upload_token, 'Upload cần HF_TOKEN có quyền ghi.'
    remote_root = ADAPTER_SUBPATH.split('/')[0] if not Path(ADAPTER_SUBPATH).is_dir() else 'local_eval'
    remote_dir = f'{remote_root}/{EVAL_OUT_DIR.name}'
    api = HfApi(token=upload_token)
    delta_msg = f'ΔPA={delta_pa:+.4f}' if delta_pa is not None else 'single-run'
    api.upload_folder(
        folder_path=str(EVAL_OUT_DIR),
        path_in_repo=remote_dir,
        repo_id=HF_REPO,
        repo_type='model',
        commit_message=f'Upload Colab eval ({delta_msg})',
    )
    print(f'✅ https://huggingface.co/{HF_REPO}/tree/main/{remote_dir}')
    if AUTO_DISCONNECT:
        from google.colab import runtime

        print('🔌 Đã lưu kết quả trên Hugging Face. Đang ngắt runtime Colab.', flush=True)
        runtime.unassign()
elif AUTO_DISCONNECT:
    print('Giữ runtime để lấy kết quả local. Bật UPLOAD_RESULTS để tự ngắt sau khi upload.')
